# Pipeline de Integração e EDA Cruzada — por UF

### Objetivo:
Integrar as bases já tratadas (INMET, IBGE Censo 2022, PNAD, EPE e Google Trends) em uma
única tabela-mestre por Unidade da Federação (UF) e realizar a análise exploratória cruzada
prevista no projeto: correlações, sazonalidade e padrões regionais.

### Saídas:
- `painel_uf_integrado.csv` — uma linha por UF, com clima, COP, domicílios, renda, consumo e interesse de busca (base da clusterização)
- `painel_mensal_uf.csv` — painel mensal (UF × ano × mês) com clima + consumo, para análise de sazonalidade

### Observação metodológica:
Todo o cruzamento é feito por **UF**, pois é a menor granularidade comum a todas as bases
públicas utilizadas (SIDRA nível 3, EPE por UF, INMET agregado por UF, Trends por estado).

In [ ]:
# ------------------------------------------------------------
# 1. MONTAGEM DO GOOGLE DRIVE
# ------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ------------------------------------------------------------
# 2. IMPORTAÇÃO DAS BIBLIOTECAS
# ------------------------------------------------------------
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ------------------------------------------------------------
# 3. CONFIGURAÇÃO
# Pasta dos dados e dimensão canônica das UFs (código IBGE -> sigla, nome, região)
# ------------------------------------------------------------
PASTA = "/content/drive/MyDrive/dados_projeto_ciencia_dados"

UF = [
    (11,"RO","Rondônia","Norte"),(12,"AC","Acre","Norte"),(13,"AM","Amazonas","Norte"),
    (14,"RR","Roraima","Norte"),(15,"PA","Pará","Norte"),(16,"AP","Amapá","Norte"),
    (17,"TO","Tocantins","Norte"),(21,"MA","Maranhão","Nordeste"),(22,"PI","Piauí","Nordeste"),
    (23,"CE","Ceará","Nordeste"),(24,"RN","Rio Grande do Norte","Nordeste"),(25,"PB","Paraíba","Nordeste"),
    (26,"PE","Pernambuco","Nordeste"),(27,"AL","Alagoas","Nordeste"),(28,"SE","Sergipe","Nordeste"),
    (29,"BA","Bahia","Nordeste"),(31,"MG","Minas Gerais","Sudeste"),(32,"ES","Espírito Santo","Sudeste"),
    (33,"RJ","Rio de Janeiro","Sudeste"),(35,"SP","São Paulo","Sudeste"),(41,"PR","Paraná","Sul"),
    (42,"SC","Santa Catarina","Sul"),(43,"RS","Rio Grande do Sul","Sul"),
    (50,"MS","Mato Grosso do Sul","Centro-Oeste"),(51,"MT","Mato Grosso","Centro-Oeste"),
    (52,"GO","Goiás","Centro-Oeste"),(53,"DF","Distrito Federal","Centro-Oeste"),
]
dim_uf   = pd.DataFrame(UF, columns=["cod_uf","uf","uf_nome","regiao"])
cod2sigla = dict(zip(dim_uf.cod_uf, dim_uf.uf))

# Meses de inverno no Brasil (hemisfério sul) — relevantes para demanda de aquecimento
MESES_INVERNO = [6, 7, 8]

In [ ]:
# ------------------------------------------------------------
# 4. CARGA DAS BASES JÁ TRATADAS
# ------------------------------------------------------------
inmet  = pd.read_csv(f"{PASTA}/inmet_mensal_por_uf.csv")
epe    = pd.read_csv(f"{PASTA}/epe_consumo_por_segmento_uf.csv")
moradores = pd.read_csv(f"{PASTA}/censo2022_moradores.csv")
tipo   = pd.read_csv(f"{PASTA}/censo2022_tipo_domicilio.csv")
agua   = pd.read_csv(f"{PASTA}/censo2022_abastecimento_agua.csv")
renda  = pd.read_csv(f"{PASTA}/pnadc_rendimento_domiciliar.csv")
trends = pd.read_csv(f"{PASTA}/gtrends_por_estado.csv")

for nome, df in [("INMET",inmet),("EPE",epe),("Moradores",moradores),
                 ("Tipo",tipo),("Água",agua),("Renda",renda),("Trends",trends)]:
    print(f"{nome:10s} {df.shape}")

In [ ]:
# ------------------------------------------------------------
# 5. INMET -> FEATURES POR UF (média anual + inverno)
# ------------------------------------------------------------
g_inmet = (
    inmet.groupby("uf").agg(
        temp_media_anual = ("temp_media_mensal",       "mean"),
        radiacao_media   = ("radiacao_kwh_m2_dia",     "mean"),
        amplitude_media  = ("amplitude_termica_diaria","mean"),
        cop_medio_anual  = ("cop_ashp_agua",           "mean"),
        aptidao_solar    = ("energia_solar_util_kwh_m2_dia","mean"),
    ).reset_index()
)
g_inverno = (
    inmet[inmet.mes.isin(MESES_INVERNO)].groupby("uf").agg(
        temp_minima_inverno  = ("temp_minima_mensal",  "mean"),
        temp_minima_absoluta = ("temp_minima_absoluta","min"),
        cop_inverno          = ("cop_ashp_agua",       "mean"),
    ).reset_index()
)
f_inmet = g_inmet.merge(g_inverno, on="uf")
print(f_inmet.head())

In [ ]:
# ------------------------------------------------------------
# 6. EPE -> CONSUMO MÉDIO MENSAL POR SEGMENTO (formato largo)
# ------------------------------------------------------------
f_epe = epe.groupby(["uf","segmento"]).consumo_mwh.mean().unstack().reset_index()
f_epe.columns = ["uf"] + [f"consumo_{c}_mwh" for c in f_epe.columns[1:]]
print(f_epe.head())

In [ ]:
# ------------------------------------------------------------
# 7. CENSO 2022 -> DOMICÍLIOS, POPULAÇÃO E PERFIL POR UF
# ------------------------------------------------------------
def valor_moradores(prefixo):
    s = moradores[moradores.variavel.str.startswith(prefixo)][["cod_uf","valor"]]
    return s.set_index("cod_uf")["valor"]

f_censo = dim_uf[["cod_uf","uf"]].copy()
f_censo["n_domicilios"]    = f_censo.cod_uf.map(valor_moradores("Domicílios particulares"))
f_censo["populacao"]       = f_censo.cod_uf.map(valor_moradores("Moradores em domicílios"))
f_censo["media_moradores"] = f_censo.cod_uf.map(valor_moradores("Média de moradores"))

# % por tipo de domicílio
tp = tipo.pivot_table(index="cod_uf", columns="categoria", values="qtd_domicilios", aggfunc="sum")
f_censo["pct_casa"]        = f_censo.cod_uf.map(tp["Casa"]        / tp["Total"])
f_censo["pct_apartamento"] = f_censo.cod_uf.map(tp["Apartamento"] / tp["Total"])

# % com ligação à rede geral de água como forma principal
ag = agua.pivot_table(index="cod_uf", columns="categoria", values="qtd_domicilios", aggfunc="sum")
f_censo["pct_rede_geral"] = f_censo.cod_uf.map(
    ag["Possui ligação à rede geral e a utiliza como forma principal"] / ag["Total"]
)
f_censo = f_censo.drop(columns="cod_uf")
print(f_censo.head())

In [ ]:
# ------------------------------------------------------------
# 8. RENDA (PNAD) e GOOGLE TRENDS -> FEATURES POR UF
# ------------------------------------------------------------
f_renda = renda[["cod_uf","rendimento_medio_pc"]].copy()
f_renda["uf"] = f_renda.cod_uf.map(cod2sigla)
f_renda = f_renda.drop(columns="cod_uf")

def slug(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode().lower()
    return "trend_" + re.sub(r"[^a-z]+","_", s).strip("_")

trp = trends.dropna(subset=["uf"]).pivot_table(
    index="uf", columns="termo", values="indice_busca").reset_index()
trp.columns = ["uf"] + [slug(c) for c in trp.columns[1:]]
print(f_renda.head()); print(trp.head())

In [ ]:
# ------------------------------------------------------------
# 9. MERGE -> TABELA-MESTRE POR UF + VARIÁVEL DERIVADA
# ------------------------------------------------------------
df_uf = (
    dim_uf
    .merge(f_inmet, on="uf")
    .merge(f_epe,   on="uf")
    .merge(f_censo, on="uf")
    .merge(f_renda, on="uf")
    .merge(trp,     on="uf")
)

# Proxy da intensidade de uso de chuveiro elétrico:
# consumo residencial médio mensal por domicílio (MWh/domicílio/mês)
df_uf["consumo_resid_por_domicilio"] = df_uf["consumo_residencial_mwh"] / df_uf["n_domicilios"]

# Oportunidade de conversão do chuveiro elétrico (alvo de mercado a substituir).
# Combina o tamanho do "prêmio" — intensidade do consumo residencial por
# domicílio — com a necessidade de aquecimento (quanto mais frio o inverno,
# maior a carga de água quente). Índice 0–100, heurístico e reponderável pelo
# time comercial. Chuveiro elétrico não é produto da empresa: é o que se busca
# substituir por coletor solar ou bomba de calor.
def _norm(s):
    return (s - s.min()) / (s.max() - s.min())

_intensidade = _norm(df_uf["consumo_resid_por_domicilio"])
_necessidade = _norm(-df_uf["temp_minima_inverno"])   # mais frio -> mais necessidade
df_uf["oportunidade_conversao"] = (100 * (0.5*_intensidade + 0.5*_necessidade)).round(1)

print("Shape:", df_uf.shape)
print("UFs:", df_uf.uf.nunique(), "| Nulos:", int(df_uf.isna().sum().sum()))
df_uf.head()

In [ ]:
# ------------------------------------------------------------
# 10. EXPORTAÇÃO DA TABELA-MESTRE
# ------------------------------------------------------------
df_uf.to_csv(f"{PASTA}/painel_uf_integrado.csv", index=False, encoding="utf-8-sig")
print("painel_uf_integrado.csv exportado.")

In [ ]:
# 12.8 — Mapa de priorização comercial: aptidão de cada produto x oportunidade
# Eixo X = aptidão solar (coletor) | Eixo Y = COP (bomba de calor)
# Tamanho do ponto = oportunidade de conversão do chuveiro elétrico
cores = {"Norte":"tab:red","Nordeste":"tab:orange","Centro-Oeste":"tab:green",
         "Sudeste":"tab:blue","Sul":"tab:purple"}
fig, ax = plt.subplots(figsize=(10,7))
for reg, g in df_uf.groupby("regiao"):
    ax.scatter(g.aptidao_solar, g.cop_medio_anual,
               s=g.oportunidade_conversao*6, color=cores[reg], alpha=0.6, label=reg)
    for _, r in g.iterrows():
        ax.annotate(r.uf, (r.aptidao_solar, r.cop_medio_anual), fontsize=8)
ax.set_xlabel("Aptidão solar — calor útil (kWh/m²/dia)")
ax.set_ylabel("COP médio anual — bomba de calor")
ax.set_title("Priorização por UF — aptidão Solar x Bomba de Calor\n(tamanho = oportunidade de conversão do chuveiro elétrico)")
ax.legend(title="Região"); plt.tight_layout(); plt.show()

# Ranking das maiores oportunidades de conversão
print("\nTop 10 UFs por oportunidade de conversão do chuveiro elétrico:")
print(df_uf.sort_values("oportunidade_conversao", ascending=False)
      [["uf","regiao","oportunidade_conversao","aptidao_solar","cop_medio_anual","rendimento_medio_pc"]]
      .head(10).to_string(index=False))

In [ ]:
# ------------------------------------------------------------
# 11. PAINEL MENSAL (UF x ano x mês) — clima + consumo
# Base para a análise de sazonalidade e correlação temperatura x consumo
# ------------------------------------------------------------
epe_wide = epe.pivot_table(
    index=["uf","ano","mes"], columns="segmento", values="consumo_mwh").reset_index()
epe_wide.columns = ["uf","ano","mes"] + [f"consumo_{c}_mwh" for c in epe_wide.columns[3:]]

painel = inmet.merge(epe_wide, on=["uf","ano","mes"], how="inner")

# Anexar nº de domicílios e calcular o consumo residencial por domicílio
# (mesma proxy do painel por UF, agora no grão mensal — usada na regressão)
painel = painel.merge(f_censo[["uf","n_domicilios"]], on="uf", how="left")
painel["consumo_resid_por_domicilio"] = (
    painel["consumo_residencial_mwh"] / painel["n_domicilios"]
)

painel.to_csv(f"{PASTA}/painel_mensal_uf.csv", index=False, encoding="utf-8-sig")
print("painel_mensal_uf.csv:", painel.shape)

## 12. Análise Exploratória Cruzada (EDA)
Estatísticas descritivas, correlações entre fontes, sazonalidade e padrões regionais.

In [ ]:
# 12.1 — Estatísticas descritivas das variáveis numéricas por UF
df_uf.describe().T

In [ ]:
# 12.2 — Heatmap de correlação entre as variáveis por UF
num = df_uf.select_dtypes("number").drop(columns=["cod_uf"])
corr = num.corr()

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                fontsize=6, color="black")
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Matriz de Correlação — variáveis por UF")
plt.tight_layout(); plt.show()

In [ ]:
# 12.3 — Dispersão: temperatura média anual x interesse por chuveiro elétrico
cores = {"Norte":"tab:red","Nordeste":"tab:orange","Centro-Oeste":"tab:green",
         "Sudeste":"tab:blue","Sul":"tab:purple"}
fig, ax = plt.subplots(figsize=(9,6))
for reg, g in df_uf.groupby("regiao"):
    ax.scatter(g.temp_media_anual, g.trend_chuveiro_eletrico,
               label=reg, color=cores[reg], s=60)
    for _, r in g.iterrows():
        ax.annotate(r.uf, (r.temp_media_anual, r.trend_chuveiro_eletrico), fontsize=8)
ax.set_xlabel("Temperatura média anual (°C)")
ax.set_ylabel("Interesse de busca — chuveiro elétrico (0–100)")
ax.set_title("Temperatura x Interesse por Chuveiro Elétrico, por UF")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 12.4 — Dispersão: COP no inverno x interesse por bomba de calor
fig, ax = plt.subplots(figsize=(9,6))
for reg, g in df_uf.groupby("regiao"):
    ax.scatter(g.cop_inverno, g.trend_bomba_de_calor, label=reg, color=cores[reg], s=60)
    for _, r in g.iterrows():
        ax.annotate(r.uf, (r.cop_inverno, r.trend_bomba_de_calor), fontsize=8)
ax.set_xlabel("COP estimado no inverno (ASHP)")
ax.set_ylabel("Interesse de busca — bomba de calor (0–100)")
ax.set_title("COP de inverno x Interesse por Bomba de Calor, por UF")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 12.5 — Sazonalidade: temperatura média x consumo residencial (Brasil)
saz = painel.groupby("mes").agg(
    temp=("temp_media_mensal","mean"),
    consumo=("consumo_residencial_mwh","mean")).reset_index()

fig, ax1 = plt.subplots(figsize=(10,5))
meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
ax1.plot(saz.mes, saz.temp, "o-", color="tab:red", label="Temp. média (°C)")
ax1.set_ylabel("Temperatura média (°C)", color="tab:red")
ax1.set_xticks(range(1,13)); ax1.set_xticklabels(meses)
ax2 = ax1.twinx()
ax2.plot(saz.mes, saz.consumo, "s--", color="tab:blue", label="Consumo residencial (MWh)")
ax2.set_ylabel("Consumo residencial médio (MWh)", color="tab:blue")
ax1.set_title("Sazonalidade — Temperatura x Consumo Residencial (média Brasil)")
plt.tight_layout(); plt.show()

In [ ]:
# 12.6 — Boxplot: consumo residencial por domicílio, por região
ordem = ["Norte","Nordeste","Centro-Oeste","Sudeste","Sul"]
dados = [df_uf[df_uf.regiao==r].consumo_resid_por_domicilio.values for r in ordem]
fig, ax = plt.subplots(figsize=(9,5))
ax.boxplot(dados)
ax.set_xticklabels(ordem)
ax.set_ylabel("Consumo residencial por domicílio (MWh/mês)")
ax.set_title("Distribuição do Consumo Residencial por Domicílio, por Região")
plt.tight_layout(); plt.show()

In [ ]:
# 12.7 — Síntese: principais correlações cruzadas de interesse para o negócio
pares = [
    ("temp_media_anual","consumo_resid_por_domicilio"),
    ("temp_minima_inverno","consumo_resid_por_domicilio"),
    ("rendimento_medio_pc","consumo_resid_por_domicilio"),
    ("rendimento_medio_pc","trend_aquecedor_solar"),
    ("temp_media_anual","trend_chuveiro_eletrico"),
    ("cop_inverno","trend_bomba_de_calor"),
    ("radiacao_media","trend_aquecimento_solar"),
    ("aptidao_solar","trend_aquecedor_solar"),
    ("oportunidade_conversao","rendimento_medio_pc"),
]
print("Correlação (Pearson) entre pares-chave por UF:\n")
for a, b in pares:
    print(f"  {a:28s} x {b:28s} = {df_uf[a].corr(df_uf[b]):+.3f}")